# Mutational Profiling PheWAS: Meta-analysis and Statistical Comparisons
**Pershad & Zhao et al., Nature Medicine, 2025**

This notebook covers the phenome-wide association study (PheWAS) analyses comparing
hazard ratios (HRs) across TET2 and DNMT3A mutation subtypes.

**Analyses:**
1. Load and process meta-analysis summary statistics for each mutation category
2. Identify phenotypes significantly associated with any mutation subtype
3. Statistical comparisons of effect size distributions (sign test, Wilcoxon, mixed-effects meta-regression)
4. Prepare data for forest plot visualization

Cox proportional hazards models were run in BioVU, All of Us, and UK Biobank separately
and combined via inverse-variance weighted fixed-effects meta-analysis (R `metafor`),
adjusted for age, age², sex, smoking, and PC1–PC10.

In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations

## 1. Load and process meta-analysis results for TET2 mutation subtypes

In [ ]:
def load_chip_meta_results(gene_prefix, categories, base_gcs_path):
    """
    Load meta-analysis summary statistics for each mutation category of a CHIP gene.
    Computes HR and 95% CI from log-scale beta coefficients.

    Parameters
    ----------
    gene_prefix   : str — gene abbreviation used in file names ('tet2' or 'd3a')
    categories    : dict — {category_label: file_suffix}
    base_gcs_path : str — GCS directory containing meta-analysis output files

    Returns
    -------
    dict of DataFrames keyed by category label
    """
    results = {}
    for label, suffix in categories.items():
        fname = f"{base_gcs_path}/meta_agescale_phewas_{gene_prefix}_{suffix}_combined2.csv" \
                if suffix else \
                f"{base_gcs_path}/meta_agescale_phewas_{gene_prefix}_combined2.csv"

        df = pd.read_csv(fname)
        df['hr']       = np.exp(df['meta_beta'])
        df['hr_upper'] = np.exp(df['meta_beta'] + 1.96 * df['meta_se'])
        df['hr_lower'] = np.exp(df['meta_beta'] - 1.96 * df['meta_se'])
        results[label] = df
        print(f"  Loaded {label}: {len(df):,} phenotypes")

    return results


base_path = "gs://bicklab-main-storage/Users/Kun_Zhao/meta_phewas_chip_mutational_pro"

# TET2: combined, pLoF (fssg = frameshift + stopgain), and missense
print("Loading TET2 meta-analysis results...")
tet2_results = load_chip_meta_results(
    gene_prefix='tet2',
    categories={'combined': '', 'pLoF': 'fssg', 'missense': 'miss'},
    base_gcs_path=base_path
)

# DNMT3A: combined, R882, and non-R882
print("\nLoading DNMT3A meta-analysis results...")
d3a_results = load_chip_meta_results(
    gene_prefix='d3a',
    categories={'combined': '', 'R882': 'R882', 'non_R882': 'non882_combined_N5'},
    base_gcs_path=base_path
)

## 2. Identify phenome-wide significant associations

We apply Bonferroni correction across all tested phenotypes.
A phenotype is included if it reaches significance in any mutation subtype.

In [ ]:
def get_significant_phenotypes(results_dict, alpha=0.05):
    """
    Identify phenotypes significant (Bonferroni-corrected) in any mutation category.
    Returns set of significant phenotype names.
    """
    sig_phenotypes = set()
    for label, df in results_dict.items():
        n_tests = len(df)
        threshold = alpha / n_tests
        sig = df.loc[df['meta_p'] < threshold, 'phenotype']
        print(f"  {label}: {len(sig)} significant phenotypes (P < {threshold:.2e})")
        sig_phenotypes |= set(sig.tolist())
    print(f"  Union of significant phenotypes: {len(sig_phenotypes)}")
    return sig_phenotypes


print("TET2 significant phenotypes:")
tet2_sig_phenos = get_significant_phenotypes(tet2_results)

print("\nDNMT3A significant phenotypes:")
d3a_sig_phenos  = get_significant_phenotypes(d3a_results)

In [ ]:
def build_comparison_table(results_dict, sig_phenotypes, gene_prefix,
                            case_label, ref_label, exclude_phenotypes=None):
    """
    Build a wide-format table of HRs and CIs for significant phenotypes,
    ready for forest plot visualization.

    Parameters
    ----------
    results_dict      : dict of DataFrames from load_chip_meta_results()
    sig_phenotypes    : set of significant phenotype names
    gene_prefix       : str — 'tet2' or 'd3a'
    case_label        : str — label for high-risk mutation category (e.g., 'pLoF')
    ref_label         : str — label for reference mutation category (e.g., 'missense')
    exclude_phenotypes: list of phenotypes to exclude from plotting
    """
    merged = None
    for label, df in results_dict.items():
        subset = df[df['phenotype'].isin(sig_phenotypes)][
            ['phenotype', 'hr', 'hr_lower', 'hr_upper', 'meta_p', 'total_samples_combined']
        ].copy()
        subset.columns = ['phenotype',
                           f'{gene_prefix}_{label}_hr',
                           f'{gene_prefix}_{label}_hr_lower',
                           f'{gene_prefix}_{label}_hr_upper',
                           f'{gene_prefix}_{label}_p',
                           f'{gene_prefix}_{label}_n']
        merged = subset if merged is None else merged.merge(subset, on='phenotype', how='outer')

    # Filter: require non-missing HRs for both mutation categories being compared
    merged = merged.dropna(subset=[f'{gene_prefix}_{case_label}_hr', f'{gene_prefix}_{ref_label}_hr'])

    # Apply phenotype exclusions (non-specific/confounded phenotypes)
    if exclude_phenotypes:
        merged = merged[~merged['phenotype'].isin(exclude_phenotypes)]

    return merged.reset_index(drop=True)


# Phenotypes excluded for non-specificity or known confounders
tet2_exclude = [
    'Sequelae of cancer', 'Abnormalities of breathing', 'Pulmonary collapse [Atelectasis]',
    'Other respiratory disorders', 'Acute posthemorrhagic anemia', 'Pain',
    'Abnormal findings on diagnostic imaging of lung', 'Dyspnea [Shortness of breath]',
    'Back pain', 'Fever', 'Constipation', 'Vomiting', 'Diarrhea'
]
d3a_exclude = [
    'Nicotine dependence (current and history of)', 'Drug-induced pancytopenia',
    'Sequelae of cancer', 'Sars-CoV-2*', 'Hemoglobinopathies'
]

tet2_phewas_df = build_comparison_table(
    tet2_results, tet2_sig_phenos, 'tet2',
    case_label='pLoF', ref_label='missense',
    exclude_phenotypes=tet2_exclude
)

d3a_phewas_df = build_comparison_table(
    d3a_results, d3a_sig_phenos, 'd3a',
    case_label='R882', ref_label='non_R882',
    exclude_phenotypes=d3a_exclude
)

print(f"TET2 phenotypes for forest plot: {len(tet2_phewas_df)}")
print(f"DNMT3A phenotypes for forest plot: {len(d3a_phewas_df)}")

tet2_phewas_df.to_csv('tet2_phewas_for_plotting.tsv', sep='\t', index=False)
d3a_phewas_df.to_csv('d3a_phewas_for_plotting.tsv',   sep='\t', index=False)

## 3. Statistical comparison of effect size distributions

We test whether pLoF HRs are systematically higher than missense HRs (TET2),
and whether R882 HRs are systematically higher than non-R882 HRs (DNMT3A),
using a sign test and one-sided Wilcoxon signed-rank test on log(HR) differences.

In [ ]:
from scipy.stats import wilcoxon, binom_test


def compare_hr_distributions(df, high_risk_hr_col, low_risk_hr_col,
                              high_risk_label='high-risk', low_risk_label='low-risk'):
    """
    Compare HR distributions between two mutation categories.
    Reports: proportion of phenotypes where high-risk HR > low-risk HR,
    sign test (one-sided binomial), and Wilcoxon signed-rank test on log(HR) differences.
    """
    log_diff = np.log(df[high_risk_hr_col]) - np.log(df[low_risk_hr_col])
    n_total  = len(log_diff)
    n_higher = (log_diff > 0).sum()

    print(f"Comparing {high_risk_label} vs. {low_risk_label}:")
    print(f"  Total phenotypes compared: {n_total}")
    print(f"  {high_risk_label} HR > {low_risk_label} HR: {n_higher}/{n_total} "
          f"({n_higher/n_total*100:.1f}%)")

    # One-sided binomial sign test (H1: HR_high > HR_low more than 50% of the time)
    p_sign = binom_test(n_higher, n_total, p=0.5, alternative='greater')
    print(f"  Sign test (one-sided) P = {p_sign:.2e}")

    # Wilcoxon signed-rank test on log(HR) differences
    stat, p_wilcox = wilcoxon(log_diff, alternative='greater')
    print(f"  Wilcoxon signed-rank (one-sided) P = {p_wilcox:.2e}")
    print(f"  Mean log(HR) difference: {log_diff.mean():.4f} (95% CI: [{log_diff.quantile(0.025):.4f}, {log_diff.quantile(0.975):.4f}])")

    return {'n_total': n_total, 'n_higher': n_higher,
            'p_sign': p_sign, 'p_wilcox': p_wilcox,
            'mean_log_diff': log_diff.mean()}


# TET2: pLoF vs. missense
tet2_comparison = compare_hr_distributions(
    tet2_phewas_df,
    high_risk_hr_col='tet2_pLoF_hr',
    low_risk_hr_col='tet2_missense_hr',
    high_risk_label='TET2 pLoF',
    low_risk_label='TET2 missense'
)

print()

# DNMT3A: R882 vs. non-R882
d3a_comparison = compare_hr_distributions(
    d3a_phewas_df,
    high_risk_hr_col='d3a_R882_hr',
    low_risk_hr_col='d3a_non_R882_hr',
    high_risk_label='DNMT3A R882',
    low_risk_label='DNMT3A non-R882'
)

## 4. Prepare for R-based forest plot and meta-regression

The mixed-effects meta-regression testing whether pLoF (or R882) mutation status
predicts systematically larger effect sizes was run in R using `metafor`.
See the companion R code block below.

In [ ]:
r_meta_regression_code = """
library(metafor)
library(data.table)

# TET2: mixed-effects meta-regression of effect size difference (pLoF vs. missense)
tet2_df <- fread('tet2_phewas_for_plotting.tsv')

# Compute log(HR) difference and pooled SE for each phenotype
tet2_df[, log_hr_diff := log(tet2_pLoF_hr) - log(tet2_missense_hr)]

# Approximate SE of log(HR) difference from CI bounds
tet2_df[, se_plof    := (log(tet2_pLoF_hr_upper)    - log(tet2_pLoF_hr_lower))    / (2 * 1.96)]
tet2_df[, se_missense := (log(tet2_missense_hr_upper) - log(tet2_missense_hr_lower)) / (2 * 1.96)]
tet2_df[, se_diff    := sqrt(se_plof^2 + se_missense^2)]

# Mixed-effects meta-regression: is log(HR) systematically higher for pLoF?
# Intercept = mean difference across phenotypes
fit_tet2 <- rma(yi = log_hr_diff, sei = se_diff, data = tet2_df, method = "REML")
summary(fit_tet2)

cat(sprintf("\nTET2 pLoF vs. Missense — Meta-regression:\n"))
cat(sprintf("  beta_diff = %.4f, SE = %.4f, P = %.2e\n",
            fit_tet2$beta, fit_tet2$se, fit_tet2$pval))
cat(sprintf("  Variance explained by mutation type: %.2f%%\n",
            fit_tet2$I2))
"""
print("Mixed-effects meta-regression code (run in R):")
print(r_meta_regression_code)